# Intro 02 — Descriptive Statistics: Summarizing Data

Practice notebook for the **"Descriptive Statistics: Summarizing Data"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — Types of Data

Statistics starts from data. Two flavors:

- **Categorical data**: labels or categories — gender, color, up/down. Arithmetic does *not* make sense.
- **Quantitative data**: numeric values where arithmetic makes sense — heights, daily returns.

The same underlying phenomenon can be recorded either way: a day's stock move is **quantitative** as a percent return, but **categorical** if we only keep the sign (up vs. down).

$$\text{data type} \;\Rightarrow\; \text{which summaries are meaningful}.$$


In [ ]:
# Same 5 trading days, recorded two ways
returns_pct = np.array([-1.2, 0.5, 1.8, -0.3, 0.7])      # quantitative
direction = np.where(returns_pct >= 0, "up", "down")     # categorical

print("Quantitative (percent return):", returns_pct.tolist())
print("Categorical  (up/down)      :", direction.tolist())

# Arithmetic on the quantitative version is meaningful:
print(f"\nMean return = {returns_pct.mean():.3f}%")
print("Mean of ['up','down'] is undefined — categorical data resists arithmetic.")

# A frequency table is the natural summary for categorical data:
vals, counts = np.unique(direction, return_counts=True)
print("\nFrequency table:")
for v, c in zip(vals, counts):
    print(f"  {v}: {c}")


**Your turn:** Convert the `returns_pct` array into a categorical *ternary* label (`up`, `flat`, `down`) using a tolerance of 0.05% around zero. How does the frequency table change?


---
# Part 2 — Measures of Central Tendency

The **sample mean** of observations \(x_1,\dots,x_n\) is

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i.$$

It is the *center of mass* of the data — and is pulled around by large values.

The **median** is the middle value when data are sorted: for odd \(n\), the central observation; for even \(n\), the average of the two central observations. The median is *robust* to outliers.

**PDF example.** For \(x=(2,3,5,10)\), \(\bar{x}=5\). Replace 10 with 100 and \(\bar{x}'=27.5\), while the median barely moves.


In [ ]:
x = np.array([2, 3, 5, 10])
x_outlier = np.array([2, 3, 5, 100])

def summarize(a):
    return (f"mean={a.mean():.3f}, median={np.median(a):.3f}")

print("Original  :", summarize(x),            " (theory mean=5, median=4)")
print("With 100  :", summarize(x_outlier),     " (theory mean=27.5, median=3.5)")

# Mean vs median on skewed data (lognormal) — mean is dragged by the long right tail
rng = np.random.default_rng(7)
skewed = rng.lognormal(mean=0.0, sigma=1.0, size=10_000)
print(f"\nLognormal sample: mean={skewed.mean():.3f}, median={np.median(skewed):.3f}")
print("Mean > median  =>  right-skewed distribution.")


**Your turn:** Draw from a *left*-skewed distribution (e.g. `rng.normal(10, 2, ...)` then square it, or use `-rng.lognormal(...)`). Confirm that now `mean < median`.


---
# Part 3 — Measures of Spread

The **sample variance** uses the \(1/(n-1)\) (Bessel) correction for unbiasedness:

$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})^2.$$

The **sample standard deviation** is \(s = \sqrt{s^2}\) — same units as the data.

**PDF example.** For \(x=(2,3,5,10)\) with \(\bar{x}=5\), the squared deviations are \(9, 4, 0, 25\), summing to 38, so \(s^2 = 38/3 \approx 12.67\) and \(s \approx 3.56\).


In [ ]:
x = np.array([2.0, 3.0, 5.0, 10.0])
xbar = x.mean()
sq_dev = (x - xbar) ** 2
ssq = sq_dev.sum()

var_bessel = ssq / (len(x) - 1)
std_bessel = np.sqrt(var_bessel)

print(f"mean (bar x)      = {xbar}")
print(f"squared deviations = {sq_dev.tolist()}")
print(f"sum sq deviations  = {ssq}  (theory 38)")
print(f"var (1/(n-1))      = {var_bessel:.4f}  (theory 12.6667)")
print(f"std dev            = {std_bessel:.4f}  (theory 3.5590)")

# Cross-check against NumPy's ddof=1 (sample variance) and scipy
print("\nNumPy  var (ddof=1):", np.var(x, ddof=1))
print("SciPy  stats.tstd   :", stats.tstd(x))
print("NumPy pop var (ddof=0):", np.var(x, ddof=0), "  <- divides by n, not n-1")


**Your turn:** As \(n\) grows, the gap between \(\frac{1}{n-1}\) and \(\frac{1}{n}\) shrinks. Simulate samples of size 10, 100, 1000 from `rng.standard_normal` and print both versions of the variance. When does the difference fall below 1%?


---
# Part 4 — Z-Scores

A **z-score** standardizes an observation \(x\) by subtracting the mean and dividing by the std:

$$z = \frac{x - \bar{x}}{s}.$$

It counts *how many standard deviations* \(x\) sits above (\(z>0\)) or below (\(z<0\)) the mean. A rule of thumb: \(|z| \gtrsim 2\) is already somewhat unusual for many distributions.

**PDF example.** With \(\bar{x}=5\) and \(s\approx 3.56\), the value \(x=10\) has \(z \approx 1.40\).


In [ ]:
x = np.array([2.0, 3.0, 5.0, 10.0])
xbar, s = x.mean(), x.std(ddof=1)
z = (x - xbar) / s
print("data     :", x.tolist())
print("z-scores :", np.round(z, 4).tolist())
print(f"\nz(10) = (10 - {xbar}) / {s:.4f} = {(10 - xbar)/s:.4f}  (theory 1.4045)")

# z-scores of a larger normal sample — empirically ~95% lie within |z| <= 2
rng = np.random.default_rng(11)
sample = rng.normal(loc=50, scale=10, size=20_000)
z_sample = (sample - sample.mean()) / sample.std(ddof=1)
within_2 = np.mean(np.abs(z_sample) <= 2)
print(f"\nN(50,10) sample: fraction with |z| <= 2 = {within_2:.4f}  (theory ~0.9545)")

# Flag potential outliers in the dataset
x_big = np.array([2.0, 3.0, 5.0, 10.0, 100.0])
z_big = (x_big - x_big.mean()) / x_big.std(ddof=1)
print(f"\nWith outlier 100: z-scores = {np.round(z_big, 3).tolist()}")
print("Note: the outlier's own z-score is NOT large, because it inflated s. "
      "Outlier detection from z-scores assumes s is not itself dominated by the outlier.")


**Your turn:** Replace the value 100 in `x_big` with 1000. Recompute the z-scores. What happens to *every* z-score? Why does this expose a weakness of z-score-based outlier flags?


---
# Part 5 — Visual Summary: Boxplot and the Five-Number Summary

A compact numerical summary is the **five-number summary**:

$$\min,\quad Q_1,\quad \text{median},\quad Q_3,\quad \max.$$

The **interquartile range** \(\text{IQR} = Q_3 - Q_1\) is a robust measure of spread, and a common outlier rule flags points below \(Q_1 - 1.5\,\text{IQR}\) or above \(Q_3 + 1.5\,\text{IQR}\). A **boxplot** visualizes exactly this.


In [ ]:
rng = np.random.default_rng(13)
clean = rng.normal(loc=0.0, scale=1.0, size=200)
data = np.concatenate([clean, [8.5, -6.0]])  # inject two outliers

q1, med, q3 = np.percentile(data, [25, 50, 75])
iqr = q3 - q1
lo_fence = q1 - 1.5 * iqr
hi_fence = q3 + 1.5 * iqr

print("Five-number summary:")
print(f"  min      = {data.min():.4f}")
print(f"  Q1       = {q1:.4f}")
print(f"  median   = {med:.4f}")
print(f"  Q3       = {q3:.4f}")
print(f"  max      = {data.max():.4f}")
print(f"  IQR      = {iqr:.4f}")
print(f"  fences   = [{lo_fence:.4f}, {hi_fence:.4f}]")
outliers = data[(data < lo_fence) | (data > hi_fence)]
print(f"  outliers = {np.round(outliers, 3).tolist()}")

fig, ax = plt.subplots()
ax.boxplot(data, vert=False)
ax.set_title("Boxplot with injected outliers")
ax.set_xlabel("value")
plt.show()

# Mean vs median in the presence of the outliers
print(f"\nmean   = {data.mean():.4f}")
print(f"median = {np.median(data):.4f}")
print("median is barely moved; mean is dragged toward the outliers.")


**Your turn:** Remove the two injected outliers from `data` and redraw the boxplot. How do the whiskers and the mean change compared to the median?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Classify each of the following as **categorical** or **quantitative**, and briefly justify: (a) a student's letter grade A/B/C/D/F; (b) the number of daily trades on an exchange; (c) a bond's credit rating AAA/AA/A…; (d) the time-to-maturity of an option in days.

2. For the dataset \(x = (4,\, 8,\, 15,\, 16,\, 23,\, 42)\), compute by hand and then verify in Python: the sample mean \(\bar{x}\), the median, the sample variance \(s^2\) (with \(1/(n-1)\)), and the sample standard deviation \(s\).

3. The dataset \((1,\, 2,\, 2,\, 3,\, 3,\, 3,\, 4,\, 4,\, 5,\, 100)\) is right-skewed. Compute the mean and the median. Which is larger, and why? What is the z-score of the value 100, and why is it *not* especially large despite 100 being an obvious outlier?

4. Generate \(10{,}000\) samples from \(N(\mu=100,\,\sigma=15)\) with a fixed seed. Report the sample mean and std, and the empirical fraction of observations with \(|z| \leq 1\), \(|z| \leq 2\), \(|z| \leq 3\). Compare to the theoretical Normal fractions (use `scipy.stats.norm`).

5. Two samples of size 50 have the *same mean* but very different spreads. Construct such samples with NumPy, draw side-by-side boxplots, and report the ratio of their sample standard deviations and the ratio of their IQRs. Which ratio better reflects the visible difference in the boxplots?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** (a) **Categorical** — letter grades are ordered labels, not meaningful numerically. (b) **Quantitative** — a count; arithmetic (averaging, differencing) makes sense. (c) **Categorical** — ordinal rating, not a quantity. (d) **Quantitative** — a measured duration in days.

**2.** \(\bar{x}=(4+8+15+16+23+42)/6 = 108/6 = 18\). Sorted, the data already is, so the median is the mean of the two middle values: \((15+16)/2 = 15.5\). Squared deviations from 18: \(196,100,9,4,25,576\), summing to 910, so \(s^2 = 910/5 = 182\) and \(s = \sqrt{182}\approx 13.49\). Python check:

```python
import numpy as np
x = np.array([4,8,15,16,23,42])
print(x.mean(), np.median(x), np.var(x, ddof=1), np.std(x, ddof=1))
# 18.0  15.5  182.0  13.4907...
```

**3.** \(\bar{x} = 12.7\), median \(= 3\). The mean is much larger than the median because the lone value 100 pulls it up — a hallmark of right-skew. The z-score of 100 is \(z = (100 - 12.7)/s\). Because 100 inflates \(s\) itself (it is the dominant contributor to the sum of squared deviations), \(s\) is large and the resulting \(z\) is modest (about 2.5 for this data). Z-scores hide outliers when the outlier drives the scale.

**4.**

```python
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)
x = rng.normal(100, 15, 10_000)
z = (x - x.mean()) / x.std(ddof=1)
for k in [1, 2, 3]:
    emp = np.mean(np.abs(z) <= k)
    th  = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"|z|<={k}: empirical {emp:.4f}, theory {th:.4f}")
# ~0.6827, ~0.9545, ~0.9973
```

**5.**

```python
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(1)
a = rng.normal(0, 1, 50)
b = rng.normal(0, 5, 50)   # same mean, much wider
print(np.std(a, ddof=1)/np.std(b, ddof=1))
iqr_a = np.diff(np.percentile(a, [25, 75]))[0]
iqr_b = np.diff(np.percentile(b, [25, 75]))[0]
print(iqr_a/iqr_b)
plt.boxplot([a, b], labels=["narrow", "wide"]); plt.show()
```

The std ratio and the IQR ratio are both near \(1/5\), matching the visibly different box widths. The IQR ratio is the more robust comparison because, unlike the std, the IQR ignores the tails.

</details>
